In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configurações do ambiente
catalog = "sptrans_catalog"
schema = "gtfs_data"

# Carrega todas as tabelasdo esquema
all_tables = spark.catalog.listTables(f"{catalog}.{schema}")

# Filtra apenas as tabelas que começam com "bronze_" e extrai o nome base
bronze_tables = [t.name.replace("bronze_", "") for t in all_tables if t.name.startswith("bronze_")]

print(f"Tabelas identificadas para processamento Silver: {bronze_tables}")

# Loop com todas as tabelas
for table_name in bronze_tables:
  print(f"Processando tabela {table_name}")

  source_table = f"{catalog}.{schema}.bronze_{table_name}"
  target_table = f"{catalog}.{schema}.silver_{table_name}"
  checkpoint_path = f"/Volumes/{catalog}/{schema}/_checkpoints/silver/silver_{table_name}"
  
  # Carrega a tabela bronze como stream
  df_silver = spark.readStream.table(source_table)

  # elimina duplicados e adiciona processing time
  df_silver = df_silver.dropDuplicates().withColumn("processing_time", F.current_timestamp())

  # Realiza os tratamentos necessários
  if table_name == 'calendar':
      df_silver = df_silver.withColumn("start_date", F.to_date(F.col("start_date").cast("string"), "yyyyMMdd"))
      df_silver = df_silver.withColumn("end_date", F.to_date(F.col("end_date").cast("string"), "yyyyMMdd"))
  elif table_name == 'fare_attributes':
      df_silver = df_silver.withColumn("transfer_duration_sec", F.col("transfer_duration").cast("int"))
      df_silver = df_silver.withColumn("transfer_duration_hours", (F.col("transfer_duration_sec") / 3600).cast("int"))
      # O INTERVAL não pode ser usado como coluna em tabelas Delta/Spark
      # df_silver = df_silver.withColumn("transfer_interval", F.expr("make_interval(0, 0, 0, 0, 0, 0, transfer_duration_sec)"))
  elif table_name == 'frequencies':
      df_silver = df_silver.withColumn(
          "headway_duration_sec", F.col("headway_secs").cast("int")
      ).withColumn(
          "headway_duration_min", (F.col("headway_duration_sec") / 60).cast("int")
      )
  elif table_name == "stop_times":
      # extraimos as partes da string (HH:mm:ss)
      time_parts = F.split(F.col("arrival_time"), ":")
      df_silver = df_silver.withColumn("arrival_hour", time_parts.getItem(0).cast("int"))
      df_silver = df_silver.withColumn("arrival_minute", time_parts.getItem(1).cast("int"))
      df_silver = df_silver.withColumn("arrival_second", time_parts.getItem(2).cast("int"))
      
      # calculamos o total de segundos desde a meia-noite
      df_silver = df_silver.withColumn(
          "arrival_time_sec",
          (
              F.col("arrival_hour") * 3600
              + F.col("arrival_minute") * 60
              + F.col("arrival_second")
          ).cast("int"),
      )
      
      # monta o formato relógio
      df_silver = df_silver.withColumn(
          "normalized_arrival_time",
          F.format_string("%02d:%02d:%02d",
                          F.col("arrival_hour") % 24,
                          F.col("arrival_minute"),
                          F.col("arrival_second")
          )  
      )

      # Faz o tratamento de string para horários
      df_silver = df_silver.withColumn("stop_sequence", F.col("stop_sequence").cast("int"))             

  # Salva na camada silver
  (df_silver.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table))
  
  print(f"Processamento da tabela {table_name} finalizado com sucesso")
